<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile
import numpy as np

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [2]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:01<00:00, 88.3MB/s]


In [3]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [4]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [5]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [6]:
len(tickers)

503

In [7]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [8]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_1338/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-29  119.264969  181.662994  133.065201  132.250000  97.580261   
2023-08-30  119.910072  185.145874  133.723358  130.610001  98.087578   
2023-08-31  118.336418  185.362946  132.497223  131.550003  96.669006   
2023-09-01  119.157463  186.931763  133.615173  132.690002  96.622032   
2023-09-05  116.049255  187.168518  131.721832  142.289993  94.771317   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-29  72.030563  306.244812  540.570007  173.920135  73.820580  ...   
2023-08-30  73.304764  306.055389  545.359985  172.976974  73.042664  ...   
2023-08-31  73.086060  306.547760  559.340027  173.177017  71.731079  ...   
2023-09-01  73.228699  310.306641  563.210022  174.622406  72.174301  ...   
2023-09-05  72.135162  308.810669  564.880005  173.770477  71.351151  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2023-08-29  3354700  1082900  4614900   9665000  1117300  7329800  1462700   
2023-08-30  2122000  1263500  2583500  11922100  1001300  6813200  1617600   
2023-08-31  4479800  2924300  3983700  17479800  2080900  7392900  1465300   
2023-09-01  3577000  1946900  2848100  14843500  1200800  5752300  1094000   
2023-09-05  4133700  3084700  4170400  18304000  2679700  5987000  1435900   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-29  2213800  280800  2423800  
2023-08-30  2018000  391000  1673500  
2023-08-31  2746000  372800  1893000  
2023-09-01  1463300  311900  1174300  
2023-09-05  1645000  417300  1673000  

[5 rows x 2515 columns]

In [9]:
df.head()

Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-29  119.264969  181.662994  133.065201  132.250000  97.580261   
2023-08-30  119.910072  185.145874  133.723358  130.610001  98.087578   
2023-08-31  118.336418  185.362946  132.497223  131.550003  96.669006   
2023-09-01  119.157463  186.931763  133.615173  132.690002  96.622032   
2023-09-05  116.049255  187.168518  131.721832  142.289993  94.771317   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-29  72.030563  306.244812  540.570007  173.920135  73.820580  ...   
2023-08-30  73.304764  306.055389  545.359985  172.976974  73.042664  ...   
2023-08-31  73.086060  306.547760  559.340027  173.177017  71.731079  ...   
2023-09-01  73.228699  310.306641  563.210022  174.622406  72.174301  ...   
2023-09-05  72.135162  308.810669  564.880005  173.770477  71.351151  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2023-08-29  3354700  1082900  4614900   9665000  1117300  7329800  1462700   
2023-08-30  2122000  1263500  2583500  11922100  1001300  6813200  1617600   
2023-08-31  4479800  2924300  3983700  17479800  2080900  7392900  1465300   
2023-09-01  3577000  1946900  2848100  14843500  1200800  5752300  1094000   
2023-09-05  4133700  3084700  4170400  18304000  2679700  5987000  1435900   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-29  2213800  280800  2423800  
2023-08-30  2018000  391000  1673500  
2023-08-31  2746000  372800  1893000  
2023-09-01  1463300  311900  1174300  
2023-09-05  1645000  417300  1673000  

[5 rows x 2515 columns]

In [10]:
closes = df["Close"]
returns = closes.pct_change() *100

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
fdf = pd.concat([cs,rs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_1338/2048977385.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return
495,2023-08-30,A,119.910072,0.540899
496,2023-08-30,AAPL,185.145874,1.917220
497,2023-08-30,ABBV,133.723358,0.494613
498,2023-08-30,ABNB,130.610001,-1.240075
499,2023-08-30,ABT,98.087578,0.519897
500,2023-08-30,ACGL,73.304764,1.768972
501,2023-08-30,ACN,306.055389,-0.061853
502,2023-08-30,ADBE,545.359985,0.886098
503,2023-08-30,ADI,172.976974,-0.542295
504,2023-08-30,ADM,73.042664,-1.053793


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [11]:
fdf.sort_values(['ticker', 'date'], inplace=True)
fdf.head(25)

,date,ticker,close,return
495,2023-08-30,A,119.910072,0.540899
990,2023-08-31,A,118.336418,-1.312362
1485,2023-09-01,A,119.157463,0.693823
1980,2023-09-05,A,116.049255,-2.608488
2475,2023-09-06,A,115.277092,-0.665376
2970,2023-09-07,A,113.635025,-1.424452
3465,2023-09-08,A,111.416267,-1.952530
3960,2023-09-11,A,110.546349,-0.780783
4455,2023-09-12,A,110.233582,-0.282928
4950,2023-09-13,A,110.868919,0.576356


In [12]:
df=fdf

In [13]:
len(df)

375506

In [14]:
df['Lag 1'] = df['close'].shift(1)
df['Lag 2'] = df['close'].shift(2)
df['Lag 3'] = df['close'].shift(3)
df['Lag 5'] = df['close'].shift(5)
df.head(200)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-08-30,A,119.910072,0.540899,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312362,119.910072,NaN,NaN,NaN
1485,2023-09-01,A,119.157463,0.693823,118.336418,119.910072,NaN,NaN
1980,2023-09-05,A,116.049255,-2.608488,119.157463,118.336418,119.910072,NaN
2475,2023-09-06,A,115.277092,-0.665376,116.049255,119.157463,118.336418,NaN
...,...,...,...,...,...,...,...,...
97349,2024-06-10,A,130.720917,0.037612,130.671768,130.514587,131.241684,129.119232
97848,2024-06-11,A,129.816879,-0.691578,130.720917,130.671768,130.514587,128.578766
98347,2024-06-12,A,130.848679,0.794811,129.816879,130.720917,130.671768,131.241684
98846,2024-06-13,A,128.303650,-1.945017,130.848679,129.816879,130.720917,130.514587


In [15]:
df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
df.head(25)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-08-30,A,119.910072,0.540899,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312362,119.910072,NaN,NaN,NaN
1485,2023-09-01,A,119.157463,0.693823,118.336418,119.910072,NaN,NaN
1980,2023-09-05,A,116.049255,-2.608488,119.157463,118.336418,119.910072,NaN
2475,2023-09-06,A,115.277092,-0.665376,116.049255,119.157463,118.336418,NaN
2970,2023-09-07,A,113.635025,-1.424452,115.277092,116.049255,119.157463,119.910072
3465,2023-09-08,A,111.416267,-1.952530,113.635025,115.277092,116.049255,118.336418
3960,2023-09-11,A,110.546349,-0.780783,111.416267,113.635025,115.277092,119.157463
4455,2023-09-12,A,110.233582,-0.282928,110.546349,111.416267,113.635025,116.049255
4950,2023-09-13,A,110.868919,0.576356,110.233582,110.546349,111.416267,115.277092


In [16]:
df['SMA 5'] = df['close'].rolling(window=5).mean()
df['SMA 10'] = df['close'].rolling(window=10).mean()
df['SMA 20'] = df['close'].rolling(window=20).mean()
df['SMA 30'] = df['close'].rolling(window=30).mean()
df['SMA 50'] = df['close'].rolling(window=50).mean()

df.head(100)
#df = fdf[fdf['ticker'] == 'AAPL'].copy()
#df.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
495,2023-08-30,A,119.910072,0.540899,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312362,119.910072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-01,A,119.157463,0.693823,118.336418,119.910072,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-05,A,116.049255,-2.608488,119.157463,118.336418,119.910072,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-06,A,115.277092,-0.665376,116.049255,119.157463,118.336418,NaN,117.746060,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47591,2024-01-17,A,125.691948,-1.846318,128.056274,128.066055,127.222389,128.203400,127.528470,128.197542,132.247807,130.859847,124.255676
48087,2024-01-18,A,127.987633,1.826437,125.691948,128.056274,128.066055,128.605682,127.404860,128.128872,131.912400,130.918633,124.679962
48583,2024-01-19,A,128.733215,0.582543,127.987633,125.691948,128.056274,127.222389,127.707025,128.150455,131.503600,131.034918,125.148346
49079,2024-01-22,A,130.037994,1.013553,128.733215,127.987633,125.691948,128.066055,128.101413,128.345686,131.238880,131.161070,125.583866


In [17]:
df['volatility 5'] = df['return'].rolling(5).std()
df['volatility 10'] = df['return'].rolling(10).std()
df['volatility 20'] = df['return'].rolling(20).std()
df['volatility 30'] = df['return'].rolling(30).std()
df.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
495,2023-08-30,A,119.910072,0.540899,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312362,119.910072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-01,A,119.157463,0.693823,118.336418,119.910072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-05,A,116.049255,-2.608488,119.157463,118.336418,119.910072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-06,A,115.277092,-0.665376,116.049255,119.157463,118.336418,NaN,117.74606,NaN,NaN,NaN,NaN,1.369003,NaN,NaN,NaN


In [18]:
df = df.dropna()
#df = df.drop(columns = 'SMA 100')
df.head()


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
24775,2023-11-08,A,107.135712,-1.040330,108.261993,105.314041,106.773331,100.740273,105.960437,103.367985,105.015814,106.415843,108.702596,2.461420,2.085483,2.164411,1.845695
25271,2023-11-09,A,105.519707,-1.508372,107.135712,108.261993,105.314041,102.317108,106.600957,103.703917,104.957050,106.284127,108.414789,2.735474,2.182338,2.005432,1.846618
25767,2023-11-10,A,106.234665,0.677559,105.519707,107.135712,108.261993,106.773331,106.493224,104.262170,104.857151,106.182107,108.172754,1.835859,2.086776,1.982966,1.852095
26263,2023-11-13,A,105.245491,-0.931122,106.234665,105.519707,107.135712,105.314041,106.479514,104.878209,104.659314,106.069803,107.894514,1.769115,2.024683,1.977988,1.856074
26759,2023-11-14,A,109.309959,3.861893,105.245491,106.234665,105.519707,108.261993,106.689107,105.685228,104.620628,106.113876,107.759729,2.200750,2.228398,2.166073,1.988012


In [19]:
len(df)

372852

In [20]:
df['future return'] = df.groupby('ticker')['return'].shift(-1)
df = df.dropna()
print(len(df))
df.head(len(df))


372349


/tmp/ipykernel_1338/2419459239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,future return
24775,2023-11-08,A,107.135712,-1.040330,108.261993,105.314041,106.773331,100.740273,105.960437,103.367985,105.015814,106.415843,108.702596,2.461420,2.085483,2.164411,1.845695,-1.508372
25271,2023-11-09,A,105.519707,-1.508372,107.135712,108.261993,105.314041,102.317108,106.600957,103.703917,104.957050,106.284127,108.414789,2.735474,2.182338,2.005432,1.846618,0.677559
25767,2023-11-10,A,106.234665,0.677559,105.519707,107.135712,108.261993,106.773331,106.493224,104.262170,104.857151,106.182107,108.172754,1.835859,2.086776,1.982966,1.852095,-0.931122
26263,2023-11-13,A,105.245491,-0.931122,106.234665,105.519707,107.135712,105.314041,106.479514,104.878209,104.659314,106.069803,107.894514,1.769115,2.024683,1.977988,1.856074,3.861893
26759,2023-11-14,A,109.309959,3.861893,105.245491,106.234665,105.519707,108.261993,106.689107,105.685228,104.620628,106.113876,107.759729,2.200750,2.228398,2.166073,1.988012,1.782994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373491,2026-08-21,ZTS,77.730003,3.612379,75.019997,76.680000,73.959999,73.790001,75.272000,74.971000,75.642500,75.445232,75.671989,2.672445,2.616024,2.685540,2.319784,-0.720449
373994,2026-08-24,ZTS,77.169998,-0.720449,77.730003,75.019997,76.680000,72.970001,76.112000,75.205000,75.655000,75.521962,75.598659,2.597205,2.517997,2.654497,2.324259,0.155504
374497,2026-08-25,ZTS,77.290001,0.155504,77.169998,77.730003,75.019997,73.959999,76.778000,75.399000,75.644000,75.646387,75.564076,2.628931,2.515249,2.649318,2.297041,0.297575
375000,2026-08-26,ZTS,77.519997,0.297575,77.290001,77.169998,77.730003,76.680000,76.945999,75.796999,75.623500,75.763252,75.526545,2.126562,2.333353,2.647344,2.295711,-3.224974


In [21]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50']
x = df[features]
y = df['future return']



In [22]:
split_index = int(0.8 * len(df))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [23]:
! pip install xgboost
import xgboost as xgb

In [24]:
model = xgb.XGBRegressor(
    n_estimators = 200,
    max_depth = 20,
    learning_rate = 0.1
)

In [25]:

from sklearn.metrics import mean_squared_error
model = model.fit(x_train, y_train)

predictions = model.predict(x_test)
error = mean_squared_error(y_test, predictions)
error

5.565227625113784

In [26]:

model = xgb.XGBRegressor(
    n_estimators = 25,
    max_depth = 10,
    learning_rate = 0.05
)

In [27]:
model = model.fit(x_train, y_train)

predictions = model.predict(x_test)
error = mean_squared_error(y_test, predictions)
error

4.981528104084157